# 基于双层优化的 Coreset 选择的实证研究

本 Notebook 实现了核心集选择方法的系统性对比实验框架。

## 对比方法
- **Random**: 随机采样基准
- **Greedy**: 贪心最远优先遍历
- **CSReL**: 基于可约损失的选择
- **BCSR**: 双层优化核心集选择
- **Ensemble**: 自适应集成方法

## 数据集
- MNIST (Split/Permuted/Rotated)
- CIFAR-10
- CIFAR-100

## 1. 环境设置

In [1]:
!nvidia-smi

Sat Apr 25 09:19:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 安装依赖
!pip install -q torch torchvision numpy scipy matplotlib seaborn pyyaml tqdm
!pip install -q pandas tensorboard

import os
import sys
import torch
import numpy as np
from pathlib import Path

print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

PyTorch版本: 2.10.0+cu128
CUDA可用: True
GPU: Tesla T4
显存: 14.56 GB


In [4]:
# 克隆仓库（如果尚未克隆）
import os

if not os.path.exists('coreset-empirical-study'):
    !git clone https://github.com/abubrak/coreset-empirical-study.git
    
%cd coreset-empirical-study
print("\n仓库结构:")
!ls -la

Cloning into 'coreset-empirical-study'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 77 (delta 21), reused 75 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 22.37 MiB | 28.78 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/coreset-empirical-study

仓库结构:
total 176
drwxr-xr-x 7 root root  4096 Apr 25 09:20 .
drwxr-xr-x 1 root root  4096 Apr 25 09:20 ..
-rw-r--r-- 1 root root 17095 Apr 25 09:20 colab_experiment.ipynb
-rw-r--r-- 1 root root  4645 Apr 25 09:20 COLAB_FIX_AND_RUN.md
-rw-r--r-- 1 root root  4655 Apr 25 09:20 COLAB_GUIDE.md
-rw-r--r-- 1 root root 22095 Apr 25 09:20 colab_simple.ipynb
-rw-r--r-- 1 root root  7332 Apr 25 09:20 colab_start.py
-rw-r--r-- 1 root root  4806 Apr 25 09:20 COLAB_USAGE.md
drwxr-xr-x 2 root root  4096 Apr 25 09:20 configs
drwxr-xr-x 3 root root  4096 Apr 25 09:20 core
drwxr-xr-x 3 root root  4096 Apr 25 09:20 data
-rw-r

In [5]:
# 导入项目模块
import sys
sys.path.insert(0, '.')

from core import get_selector, ContinualLearningFramework
from data.datasets import create_dataloaders

print("✅ 模块导入成功")

✅ 模块导入成功


## 2. 快速验证

先测试所有方法是否正常工作。

In [6]:
# 运行快速验证
!python run_quick.py

快速验证测试
设备: cuda
/content/coreset-empirical-study/data/datasets.py:95: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  targets = torch.tensor(self.full_dataset.targets)
数据集: MNIST, 10 类

--- 测试 random ---
  选择 500 样本, 权重范围 [1.0000, 1.0000]
  [OK] random 通过

--- 测试 greedy ---
  选择 500 样本, 权重范围 [1.0000, 1.0000]
  [OK] greedy 通过

--- 测试 csrel ---
  选择 500 样本, 权重范围 [0.0000, 0.0001]
  [OK] csrel 通过

--- 测试 bcsr ---
  [FAIL] bcsr 失败: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.
Traceback (most recent call last):
  File "/content/cores

## 3. 运行对比实验

### 3.1 快速模式（推荐）

在 MNIST 上运行，减少 epochs 和运行次数，适合快速验证。

In [ ]:
# 快速实验：MNIST，10 epochs，单次运行
!python experiments/run_comparison.py \
    --dataset mnist \
    --method random greedy csrel bcsr \
    --quick

print("\n✅ 快速实验完成！")

### 3.2 完整实验（可选）

运行完整实验矩阵，需要较长时间。建议在 GPU 环境下运行。

In [ ]:
# 完整实验：多数据集、多方法、多次运行
# 取消注释以运行（预计需要 1-2 小时）

# !python experiments/run_comparison.py --config configs/experiments.yaml

print("完整实验已注释，如需运行请取消注释")

### 3.3 自定义实验

你可以自定义实验参数：

In [ ]:
# 自定义实验示例
!python experiments/run_comparison.py \
    --dataset mnist cifar10 \
    --method csrel bcsr ensemble \
    --config configs/experiments.yaml

print("\n自定义实验完成！")

## 4. 结果分析与可视化

生成对比图表和统计表格。

In [ ]:
# 查找最新的实验结果
import glob

result_files = glob.glob('results/comparison_*.json')
if result_files:
    latest_result = max(result_files)
    print(f"最新结果文件: {latest_result}")
else:
    print("未找到实验结果，请先运行实验")
    latest_result = None

In [ ]:
# 生成分析图表
if latest_result:
    !python experiments/analysis.py "{latest_result}"
    print(f"\n✅ 分析完成！图表保存在 results/figures/")
else:
    print("请先运行实验")

## 5. 查看结果

In [ ]:
# 显示汇总表格
import pandas as pd

try:
    df = pd.read_csv('results/figures/summary_table.csv')
    print("\n=== 实验结果汇总 ===\n")
    print(df.to_string(index=False))
except:
    print("未找到汇总表格")

In [ ]:
# 显示生成的图表
from IPython.display import Image, display
import glob

figures = glob.glob('results/figures/*.png')
if figures:
    print(f"\n找到 {len(figures)} 张图表:\n")
    for fig in sorted(figures):
        print(f"📊 {fig.split('/')[-1]}")
        display(Image(filename=fig, width=600))
        print()
else:
    print("未找到图表文件")

## 6. 交互式分析

直接在 Notebook 中进行数据分析。

In [ ]:
# 加载实验结果
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

with open(latest_result, 'r') as f:
    results = json.load(f)

print(f"加载了 {len(results)} 组实验结果")

In [ ]:
# 方法性能对比
method_names = {
    'random': 'Random',
    'greedy': 'Greedy',
    'csrel': 'CSReL',
    'bcsr': 'BCSR',
    'ensemble': 'Ensemble'
}

# 按方法分组
method_performance = {}
for r in results:
    method = r['method']
    if method not in method_performance:
        method_performance[method] = []
    method_performance[method].append(r['summary']['average_accuracy'])

# 计算平均性能
df_performance = pd.DataFrame([
    {'Method': method_names[m], 'Avg Accuracy': np.mean(v), 'Std': np.std(v)}
    for m, v in method_performance.items()
]).sort_values('Avg Accuracy', ascending=False)

print("\n=== 方法排名（按平均准确率）===\n")
print(df_performance.to_string(index=False))

# 可视化
plt.figure(figsize=(10, 6))
sns.barplot(data=df_performance, x='Method', y='Avg Accuracy')
plt.title('Average Accuracy by Method')
plt.ylabel('Average Accuracy')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# 遗忘度量分析
df_forgetting = pd.DataFrame([
    {'Method': method_names[m], 'Forgetting': np.mean([r['summary']['forgetting_measure'] for r in results if r['method'] == m])}
    for m in method_performance.keys()
]).sort_values('Forgetting')

print("\n=== 遗忘度量（越低越好）===\n")
print(df_forgetting.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=df_forgetting, x='Method', y='Forgetting')
plt.title('Forgetting Measure by Method (Lower is Better)')
plt.ylabel('Forgetting Measure')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 7. 下载结果

将实验结果和图表下载到本地。

In [ ]:
# 打包结果文件
!zip -r results.zip results/
print("\n结果已打包为 results.zip")

In [ ]:
# 生成下载链接
from google.colab import files

files.download('results.zip')
print("\n✅ 下载完成！")

## 8. 实验总结

### 完成的工作
- ✅ 实现了 5 种核心集选择方法的统一对比框架
- ✅ 在持续学习场景下进行系统性实验
- ✅ 生成准确率、遗忘度量、敏感性等多维度分析

### 主要发现
- 根据实验结果，分析各方法的优缺点
- 双层优化方法（BCSR）与传统方法的性能差异
- 不同核心集大小下的方法敏感性

### 论文贡献
1. 提供了持续学习中核心集选择的统一对比平台
2. 系统分析了简化实现与原始方法的性能差距
3. 为实际应用提供了可行的方法选择指导

---

**注意**：本实验中的 BCSR 和 CSReL 实现为简化版本，详细说明请参阅 `IMPLEMENTATION_ANALYSIS.md`。

**项目仓库**：https://github.com/abubrak/coreset-empirical-study